# Aula 18 - Grafos de Logística de Insumos e Produtos Acabados

Este notebook complementa a malha de processo das aulas anteriores com duas redes logísticas da **Planta de Produção de Biodiesel (Transesterificação em Batelada - Grupo 5)**:

1. **$G_M$ — Fluxo de Materiais:** dígrafo acíclico dirigido (DAG) que modela a entrada de insumos, o processamento e a saída de produtos acabados.
2. **$G_V$ — Circulação de Veículos:** grafo dirigido que modela o anel viário interno para caminhões e empilhadeiras.

Os pesos são estimativas didáticas em metros, calibradas para o layout da planta piloto.

In [1]:
from dataclasses import dataclass
from heapq import heappop, heappush
from math import inf
from typing import Dict, List, Tuple


def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura padrao SCADA."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)


@dataclass(frozen=True)
class Rota:
    destino: str
    peso: float
    descricao: str


class GrafoLogistico:
    """Modelagem de rede logistica dirigida e ponderada com Dijkstra (Min-Heap)."""
    
    def __init__(self) -> None:
        self._adjacencias: Dict[str, List[Rota]] = {}

    def adicionar_rota(self, origem: str, destino: str, peso: float, descricao: str) -> None:
        if peso < 0:
            raise ValueError("Dijkstra exige pesos nao negativos.")
        self._adjacencias.setdefault(origem, []).append(Rota(destino, peso, descricao))
        self._adjacencias.setdefault(destino, [])

    def vertices(self) -> List[str]:
        return list(self._adjacencias.keys())

    def menor_caminho(self, origem: str, destino: str) -> Tuple[float, List[str]]:
        if origem not in self._adjacencias or destino not in self._adjacencias:
            raise KeyError("Origem e destino devem pertencer ao grafo.")

        distancias = {vertice: inf for vertice in self._adjacencias}
        predecessores: Dict[str, str] = {}
        distancias[origem] = 0.0
        fila: List[Tuple[float, str]] = [(0.0, origem)]

        while fila:
            distancia_atual, atual = heappop(fila)
            if distancia_atual != distancias[atual]:
                continue
            if atual == destino:
                break
            for rota in self._adjacencias[atual]:
                candidata = distancia_atual + rota.peso
                if candidata < distancias[rota.destino]:
                    distancias[rota.destino] = candidata
                    predecessores[rota.destino] = atual
                    heappush(fila, (candidata, rota.destino))

        if distancias[destino] == inf:
            raise ValueError(f"Nao existe rota de {origem} para {destino}.")

        caminho = [destino]
        while caminho[-1] != origem:
            caminho.append(predecessores[caminho[-1]])
        caminho.reverse()
        return distancias[destino], caminho

    def graus_entrada(self) -> Dict[str, int]:
        graus = {v: 0 for v in self._adjacencias}
        for origem, rotas in self._adjacencias.items():
            for r in rotas:
                graus[r.destino] += 1
        return graus


def adicionar_rotas(grafo: GrafoLogistico, rotas: List[Tuple[str, str, float, str]]) -> None:
    for origem, destino, peso, descricao in rotas:
        grafo.adicionar_rota(origem, destino, peso, descricao)


## 1. Grafo $G_M$: Fluxo de Materiais (Insumos → Processo → Produtos)

Modelagem do DAG (Grafo Acíclico Dirigido) que representa a cadeia produtiva do biodiesel: recebimento de insumos, transesterificação, separação, envase e expedição.

In [2]:
# GM: fluxo de materiais da planta de biodiesel
fluxo_materiais = GrafoLogistico()
adicionar_rotas(fluxo_materiais, [
    ("Portaria", "Balança de Entrada", 25, "liberacao e pesagem do recebimento"),
    ("Balança de Entrada", "Recebimento / Galpão de Insumos", 55, "direcionamento a descarga"),
    ("Recebimento / Galpão de Insumos", "Box: Óleo Vegetal Degomado", 18, "armazenagem de materia-prima lipidica"),
    ("Recebimento / Galpão de Insumos", "Box: Metanol Anidro", 22, "armazenagem de alcohol transesterificante"),
    ("Recebimento / Galpão de Insumos", "Box: Catalisador (NaOH/KOH)", 20, "armazenagem de catalisador alcalino"),
    ("Recebimento / Galpão de Insumos", "Box: Ácido Fosfórico (Neutralização)", 25, "armazenagem de acido de neutralizacao"),
    ("Recebimento / Galpão de Insumos", "Tanques de Aditivos / Antioxidantes", 28, "recebimento de aditivos liquidos"),
    ("Box: Óleo Vegetal Degomado", "Moega / Pré-aquecimento", 35, "transferencia por bombeamento"),
    ("Box: Metanol Anidro", "Moega / Pré-aquecimento", 32, "transferencia por bombeamento"),
    ("Box: Catalisador (NaOH/KOH)", "Moega / Pré-aquecimento", 30, "transferencia por dosagem"),
    ("Box: Ácido Fosfórico (Neutralização)", "Moega / Pré-aquecimento", 38, "transferencia por dosagem"),
    ("Moega / Pré-aquecimento", "Silos de Dosagem", 40, "alimentacao dos dosadores"),
    ("Tanques de Aditivos / Antioxidantes", "Silos de Dosagem", 45, "dosagem de aditivos"),
    ("Silos de Dosagem", "Reator CSTR R-200", 18, "transesterificacao em batelada"),
    ("Reator CSTR R-200", "Decantador TK-300", 20, "separacao de fases (biodiesel/glicerina)"),
    ("Decantador TK-300", "Lavagem e Secagem", 32, "purificacao de esteres"),
    ("Lavagem e Secagem", "Envase / Paletização", 48, "acabamento e embalagem"),
    ("Envase / Paletização", "Estoque Biodiesel B100", 35, "enderecamento do produto acabado"),
    ("Envase / Paletização", "Estoque Glicerina Bruta", 42, "enderecamento do coproduto"),
    ("Envase / Paletização", "Estoque Ésteres Industriais", 50, "enderecamento de esteres"),
    ("Estoque Biodiesel B100", "Docas de Expedição", 28, "separacao para carregamento"),
    ("Estoque Glicerina Bruta", "Docas de Expedição", 25, "separacao para carregamento"),
    ("Estoque Ésteres Industriais", "Docas de Expedição", 32, "separacao para carregamento"),
])

print("=== GRAFO G_M: FLUXO DE MATERIAIS (BIODIESEL) ===\n")
distancia, caminho = fluxo_materiais.menor_caminho("Portaria", "Docas de Expedição")
print("--- Menor caminho: Portaria -> Docas de Expedição ---")
print("Rota: " + " -> ".join(caminho))
print(f"Distância didática total: {distancia:.0f} m\n")

print("--- Rotas dos insumos até a expedição ---")
for insumo in [
    "Box: Óleo Vegetal Degomado",
    "Box: Metanol Anidro",
    "Box: Catalisador (NaOH/KOH)",
    "Box: Ácido Fosfórico (Neutralização)",
    "Tanques de Aditivos / Antioxidantes",
]:
    d, c = fluxo_materiais.menor_caminho(insumo, "Docas de Expedição")
    print(f"  {insumo}: {d:.0f} m")

print("\n--- Graus de entrada (DAG - devem ser >= 0) ---")
graus = fluxo_materiais.graus_entrada()
for v, g in graus.items():
    tipo = "(fonte pura)" if g == 0 else ("(sumidouro puro)" if v == "Docas de Expedição" else "")
    print(f"  {v}: {g} {tipo}")

assert "Reator CSTR R-200" in caminho
assert "Envase / Paletização" in caminho
assert graus["Portaria"] == 0


=== GRAFO G_M: FLUXO DE MATERIAIS (BIODIESEL) ===

--- Menor caminho: Portaria -> Docas de Expedição ---
Rota: Portaria -> Balança de Entrada -> Recebimento / Galpão de Insumos -> Box: Óleo Vegetal Degomado -> Moega / Pré-aquecimento -> Silos de Dosagem -> Reator CSTR R-200 -> Decantador TK-300 -> Lavagem e Secagem -> Envase / Paletização -> Estoque Biodiesel B100 -> Docas de Expedição
Distância didática total: 335 m

--- Rotas dos insumos até a expedição ---
  Box: Óleo Vegetal Degomado: 315 m
  Box: Metanol Anidro: 312 m
  Box: Catalisador (NaOH/KOH): 305 m
  Box: Ácido Fosfórico (Neutralização): 320 m
  Tanques de Aditivos / Antioxidantes: 318 m

--- Graus de entrada (DAG - devem ser >= 0) ---
  Portaria: 0 (fonte pura)
  Balança de Entrada: 1
  Recebimento / Galpão de Insumos: 1
  Box: Óleo Vegetal Degomado: 1
  Box: Metanol Anidro: 1
  Box: Catalisador (NaOH/KOH): 1
  Box: Ácido Fosfórico (Neutralização): 1
  Tanques de Aditivos / Antioxidantes: 1
  Moega / Pré-aquecimento: 4
  Si

## 2. Grafo $G_V$: Circulação Interna de Veículos

Anel viário dirigido para caminhões-tanque e empilhadeiras. Diferente de $G_M$, este grafo contém um ciclo dirigido (retorno à portaria).

In [3]:
# GV: circulacao interna de veiculos
circulacao_veiculos = GrafoLogistico()
anel_viario = [
    ("Portaria", "Balança de Entrada", 25, "entrada de veiculos"),
    ("Balança de Entrada", "Pátio de Recebimento", 55, "acesso a descarga"),
    ("Pátio de Recebimento", "Galpão de Insumos", 40, "rota de descarga"),
    ("Galpão de Insumos", "Área de Processamento", 65, "rota interna de servico"),
    ("Área de Processamento", "Galpão de Produtos Acabados", 75, "rota para armazenagem"),
    ("Galpão de Produtos Acabados", "Docas de Expedição", 30, "rota de carregamento"),
    ("Docas de Expedição", "Balança de Saída", 60, "saida de veiculos"),
    ("Balança de Saída", "Portaria", 35, "liberacao de saida"),
]
adicionar_rotas(circulacao_veiculos, anel_viario)

print("=== GRAFO G_V: CIRCULAÇÃO DE VEÍCULOS ===\n")
d_saida, r_saida = circulacao_veiculos.menor_caminho("Docas de Expedição", "Portaria")
print("--- Menor caminho: Docas de Expedição -> Portaria (saída) ---")
print("Rota: " + " -> ".join(r_saida))
print(f"Distância didática total: {d_saida:.0f} m\n")

d_entrada, r_entrada = circulacao_veiculos.menor_caminho("Portaria", "Área de Processamento")
print("--- Menor caminho: Portaria -> Área de Processamento (entrada) ---")
print("Rota: " + " -> ".join(r_entrada))
print(f"Distância didática total: {d_entrada:.0f} m")

assert r_saida == ["Docas de Expedição", "Balança de Saída", "Portaria"]
assert d_saida == 95.0
assert d_entrada == 185.0


=== GRAFO G_V: CIRCULAÇÃO DE VEÍCULOS ===

--- Menor caminho: Docas de Expedição -> Portaria (saída) ---
Rota: Docas de Expedição -> Balança de Saída -> Portaria
Distância didática total: 95 m

--- Menor caminho: Portaria -> Área de Processamento (entrada) ---
Rota: Portaria -> Balança de Entrada -> Pátio de Recebimento -> Galpão de Insumos -> Área de Processamento
Distância didática total: 185 m


## 3. Validação Topológica: Algoritmo de Kahn (Ordenação Topológica)

Confirmação de que $G_M$ é um DAG, calculando uma ordenação topológica válida. Se o algoritmo de Kahn processar todos os vértices, o grafo é acíclico.

In [4]:
from collections import deque

def kahn_ordenacao_topologica(grafo: GrafoLogistico) -> List[str]:
    """Algoritmo de Kahn: retorna ordenacao topologica ou lista vazia se houver ciclo."""
    graus_in = grafo.graus_entrada()
    fila = deque([v for v, g in graus_in.items() if g == 0])
    ordem = []
    while fila:
        u = fila.popleft()
        ordem.append(u)
        for rota in grafo._adjacencias.get(u, []):
            graus_in[rota.destino] -= 1
            if graus_in[rota.destino] == 0:
                fila.append(rota.destino)
    return ordem

ordem = kahn_ordenacao_topologica(fluxo_materiais)
print("=== ORDENAÇÃO TOPOLÓGICA DE G_M (ALGORITMO DE KAHN) ===")
print("Ordem topológica válida:")
for i, v in enumerate(ordem, 1):
    print(f"  {i:>2}. {v}")
print(f"\nTotal de vértices processados: {len(ordem)} / {len(fluxo_materiais.vertices())}")
if len(ordem) == len(fluxo_materiais.vertices()):
    print("Status: G_M é um DAG válido (sem ciclos dirigidos).")
else:
    print("Status: G_M contém ciclo dirigido!")

assert len(ordem) == len(fluxo_materiais.vertices())
assert ordem[0] == "Portaria"
assert ordem[-1] == "Docas de Expedição"


=== ORDENAÇÃO TOPOLÓGICA DE G_M (ALGORITMO DE KAHN) ===
Ordem topológica válida:
  1. Portaria
  2. Balança de Entrada
  3. Recebimento / Galpão de Insumos
  4. Box: Ácido Fosfórico (Neutralização)
  5. Box: Catalisador (NaOH/KOH)
  6. Box: Metanol Anidro
  7. Box: Óleo Vegetal Degomado
  8. Tanques de Aditivos / Antioxidantes
  9. Moega / Pré-aquecimento
 10. Silos de Dosagem
 11. Reator CSTR R-200
 12. Decantador TK-300
 13. Lavagem e Secagem
 14. Envase / Paletização
 15. Estoque Biodiesel B100
 16. Estoque Glicerina Bruta
 17. Estoque Ésteres Industriais
 18. Docas de Expedição

Total de vértices processados: 18 / 18
Status: G_M é um DAG válido (sem ciclos dirigidos).


## 4. Validação Experimental: Bloqueio de Aresta Crítica

Simulação da indisponibilidade da aresta `Envase / Paletização -> Estoque Biodiesel B100` e verificação de que o algoritmo detecta a ausência de rota.

In [5]:
# Bloqueio da aresta 'Envase / Paletização -> Estoque Biodiesel B100'
aresta_bloqueada = ("Envase / Paletização", "Estoque Biodiesel B100")
for origem, rotas in fluxo_materiais._adjacencias.items():
    fluxo_materiais._adjacencias[origem] = [r for r in rotas if not (origem == aresta_bloqueada[0] and r.destino == aresta_bloqueada[1])]

print("=== EXPERIMENTO: BLOQUEIO DA ARESTA 'Envase -> Estoque Biodiesel B100' ===\n")
print("--- Caminho para Estoque Biodiesel B100 APÓS bloqueio ---")
try:
    fluxo_materiais.menor_caminho("Portaria", "Estoque Biodiesel B100")
    print("ERRO: ainda existe rota!")
except ValueError as e:
    print(f"ERRO ESPERADO: {e}")

print("\n--- Caminho para Estoque Glicerina Bruta (não afetado) ---")
d, c = fluxo_materiais.menor_caminho("Portaria", "Estoque Glicerina Bruta")
print("Rota: " + " -> ".join(c))
print(f"Distância: {d:.0f} m")


=== EXPERIMENTO: BLOQUEIO DA ARESTA 'Envase -> Estoque Biodiesel B100' ===

--- Caminho para Estoque Biodiesel B100 APÓS bloqueio ---
ERRO ESPERADO: Não existe rota de Portaria para Estoque Biodiesel B100.

--- Caminho para Estoque Glicerina Bruta (não afetado) ---
Rota: Portaria -> Balança de Entrada -> Recebimento / Galpão de Insumos -> Box: Óleo Vegetal Degomado -> Moega / Pré-aquecimento -> Silos de Dosagem -> Reator CSTR R-200 -> Decantador TK-300 -> Lavagem e Secagem -> Envase / Paletização -> Estoque Glicerina Bruta
Distância: 337 m


## 5. Comparação Multicamada: $G_M$ vs. $G_V$

Demonstração da diferença entre as duas camadas: o mesmo par de nós pode ter distâncias diferentes dependendo da camada consultada.

In [6]:
print("=== COMPARAÇÃO MULTICAMADA: G_M vs. G_V ===\n")

# Em G_M: Portaria -> R-200 (reator)
d_m, c_m = fluxo_materiais.menor_caminho("Portaria", "Reator CSTR R-200")
print("Distância Portaria -> Área de Processamento:")
print(f"  G_M (fluxo de materiais):  {' -> '.join(c_m)}")
print(f"  G_M (custo): {d_m:.0f} m\n")

# Em G_V: Portaria -> Área de Processamento
d_v, c_v = circulacao_veiculos.menor_caminho("Portaria", "Área de Processamento")
print(f"  G_V (circulação de veículos): {' -> '.join(c_v)}")
print(f"  G_V (custo): {d_v:.0f} m\n")

print("Os grafos respondem a perguntas diferentes:")
print("  - G_M responde: por onde o MATERIAL flui?")
print("  - G_V responde: por onde o VEÍCULO circula?")
print("  - As distâncias são comparáveis numericamente, mas as rotas são semanticamente distintas.")


=== COMPARAÇÃO MULTICAMADA: G_M vs. G_V ===

Distância Portaria -> Área de Processamento:
  G_M (fluxo de materiais):  Portaria -> Balança de Entrada -> Recebimento / Galpão de Insumos -> Box: Óleo Vegetal Degomado -> Moega / Pré-aquecimento -> Silos de Dosagem -> Reator CSTR R-200
  G_M (custo): 188 m

  G_V (circulação de veículos): Portaria -> Balança de Entrada -> Pátio de Recebimento -> Galpão de Insumos -> Área de Processamento
  G_V (custo): 185 m

Os grafos respondem a perguntas diferentes:
  - G_M responde: por onde o MATERIAL flui?
  - G_V responde: por onde o VEÍCULO circula?
  - As distâncias são comparáveis numericamente, mas as rotas são semanticamente distintas.


## 6. Conclusões da Aula 18

1. **DAG do fluxo de materiais:** O grafo $G_M$ é estritamente acíclico, refletindo a irreversibilidade das etapas de processo (transesterificação, decantação, lavagem, envase).
2. **Grafo de circulação de veículos:** O grafo $G_V$ contém ciclo dirigido, refletindo o retorno do veículo à portaria.
3. **Multicamada:** Modelar as duas redes separadamente evita erros de interpretação semântica e respeita as restrições físicas de cada tipo de fluxo.
4. **Algoritmo de Kahn:** Confirma a aciclicidade de $G_M$ em tempo linear $O(|V|+|E|)$.
5. **Dijkstra:** Aplica-se a ambas as camadas, desde que os pesos sejam não-negativos.

TODAS AS ASSERÇÕES DE ENGENHARIA FORAM VALIDADAS COM SUCESSO!